# 04 — Test Execution Demo (Playwright Version)

This notebook executes the Playwright test script generated in Notebook 03.

Pipeline so far:
NL → IR → Mapping → Code → **Execution** → Artifacts → Analysis

This notebook:
- Loads the generated test file
- Executes it using Playwright
- Captures artifacts (screenshots, logs, trace)
- Saves results for Notebook 05


In [ ]:
import subprocess
import json
from pathlib import Path
import shutil
import datetime


## 4.1 — Paths and Artifact Setup

We prepare directories for:
- generated test file
- execution artifacts
- trace files

In [ ]:
test_file = Path("../generated-tests/todomvc_test.py")

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
artifact_dir = Path(f"../artifacts/run_{timestamp}")
artifact_dir.mkdir(parents=True, exist_ok=True)

screenshot_path = artifact_dir / "screenshot.png"
trace_path = artifact_dir / "trace.zip"
log_path = artifact_dir / "execution_log.txt"

artifact_dir


## 4.2 — Wrap the Test File to Capture Artifacts

We modify the generated test to:
- take a screenshot at the end
- start/stop Playwright tracing
- save artifacts to our folder


In [ ]:
wrapped_test_path = artifact_dir / "wrapped_test.py"

with open(test_file, "r") as f:
    original_code = f.read()

wrapped_code = f"""
from playwright.async_api import async_playwright, expect
import asyncio

async def run_test():
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=True)
        context = await browser.new_context()
        await context.tracing.start(screenshots=True, snapshots=True, sources=True)
        page = await context.new_page()
        await page.goto("https://demo.playwright.dev/todomvc")

{original_code.split("async def run_test():")[1].split("if __name__")[0]}

        await page.screenshot(path="{screenshot_path.as_posix()}")
        await context.tracing.stop(path="{trace_path.as_posix()}")
        await browser.close()

if __name__ == "__main__":
    asyncio.run(run_test())
"""

with open(wrapped_test_path, "w") as f:
    f.write(wrapped_code)

wrapped_test_path


## 4.3 — Execute the Wrapped Test

We run the Playwright script as a subprocess and capture:
- stdout
- stderr
- return code


In [ ]:
result = subprocess.run(
    ["python", wrapped_test_path],
    capture_output=True,
    text=True
)

execution_output = {
    "return_code": result.returncode,
    "stdout": result.stdout,
    "stderr": result.stderr,
    "artifacts": {
        "screenshot": screenshot_path.as_posix(),
        "trace": trace_path.as_posix(),
        "log": log_path.as_posix()
    }
}

execution_output


## 4.4 — Save Execution Log

We store stdout/stderr for Notebook 05 to analyze.


In [ ]:
with open(log_path, "w") as f:
    f.write("STDOUT:\n")
    f.write(result.stdout)
    f.write("\n\nSTDERR:\n")
    f.write(result.stderr)

log_path


## 4.5 — Save Execution Summary

Notebook 05 will load this JSON file.


In [ ]:
summary_path = artifact_dir / "execution_summary.json"

with open(summary_path, "w") as f:
    json.dump(execution_output, f, indent=4)

summary_path


## Summary

In this notebook we:

- Loaded the generated Playwright test file
- Wrapped it to capture screenshots and traces
- Executed it in a real Chromium browser
- Saved artifacts (screenshot, trace, logs)
- Saved an execution summary for Notebook 05

Next step: analyze the artifacts in Notebook 05.
